<a href="https://colab.research.google.com/github/yerin08/hankyung-toss-fullstack/blob/main/data-structures/hash_indexing_and_performance_comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **고급 해시 사용 연습**

# PART I

## 1. 실습명 : CSV 파일 기반 해시 인덱스 생성과 검색 성능 비교

대용량 CSV·로그 파일을 DB 없이 직접 처리해야 하는 배치 및 분석 환경에서 사용된다. 파일 전체를 반복 탐색하는 구조는 비효율적이므로 메모리 기반 인덱스를 통해 성능을 개선한다. 해시 인덱스는 특정 ID 조회를 즉시 처리할 수 있어 반복 검색 상황에서 효과적이다. DB 인덱스의 내부 원리를 코드 수준에서 이해하고 적용하기 위한 목적

## 2. 실습 목적

- Node.js에서 실제 CSV 파일을 생성하고 읽는 과정을 통해 파일 기반 데이터 처리 구조를 이해.
- 선형 탐색과 해시 인덱스 탐색을 각각 구현하여 검색 방식에 따른 실행 속도 차이를 확인.
- 인덱스 생성 비용과 검색 성능 향상의 관계를 이해.
- 데이터베이스 인덱스의 기본 원리를 단순한 코드로 학습.

## 3. 실습 개요

특정 회원 ID를 찾는 기능을 두 가지 방식으로 구하고 두 방식의 실행 시간을 측정하여 검색 구조의 차이를 비교

- 선형 탐색
- 해시 인덱스 탐색

In [5]:
%%writefile app.js

const fs = require('fs');
const path = require('path');

const FILE_PATH = path.join(__dirname, 'users.csv');
const TARGET_ID = 'user0300000';
const SEARCH_REPEAT = 10;

function readCsvLines() {
    if (!fs.existsSync(FILE_PATH)) {
        throw new Error('users.csv 파일이 없습니다.');
    }

    const content = fs.readFileSync(FILE_PATH, 'utf8');
    return content.trim().split('\n');
}

function parseCsvToArray() {
    const lines = readCsvLines();
    const header = lines[0].split(',');

    const rows = [];

    for (let i = 1; i < lines.length; i++) {
        const cols = lines[i].split(',');
        rows.push({
            [header[0]]: cols[0],
            [header[1]]: cols[1],
            [header[2]]: Number(cols[2]),
            [header[3]]: cols[3]
        });
    }

    return rows;
}

function buildHashIndex(rows) {
    const index = Object.create(null);

    for (let i = 0; i < rows.length; i++) {
        index[rows[i].user_id] = rows[i];
    }

    return index;
}

function linearSearchFromFile(targetId) {
    const lines = readCsvLines();

    for (let i = 1; i < lines.length; i++) {
        const cols = lines[i].split(',');

        if (cols[0] === targetId) {
            return {
                user_id: cols[0],
                name: cols[1],
                age: Number(cols[2]),
                city: cols[3]
            };
        }
    }

    return null;
}

function linearSearchFromRows(rows, targetId) {
    for (let i = 0; i < rows.length; i++) {
        if (rows[i].user_id === targetId) {
            return rows[i];
        }
    }

    return null;
}

function hashSearch(index, targetId) {
    return index[targetId] || null;
}

function measureIndexBuildCost(rows) {
    const start = process.hrtime.bigint();
    const index = buildHashIndex(rows);
    const end = process.hrtime.bigint();

    console.log(`해시 인덱스 생성 시간: ${Number(end - start) / 1000000} ms`);
    return index;
}

function measureSingleSearch(rows, index) {
    let start;
    let end;
    let result;

    start = process.hrtime.bigint();
    result = linearSearchFromFile(TARGET_ID);
    end = process.hrtime.bigint();
    console.log(`파일 직접 선형 탐색 시간: ${Number(end - start) / 1000000} ms`);

    start = process.hrtime.bigint();
    linearSearchFromRows(rows, TARGET_ID);
    end = process.hrtime.bigint();
    console.log(`메모리 배열 선형 탐색 시간: ${Number(end - start) / 1000000} ms`);

    start = process.hrtime.bigint();
    hashSearch(index, TARGET_ID);
    end = process.hrtime.bigint();
    console.log(`해시 인덱스 탐색 시간: ${Number(end - start) / 1000000} ms`);

    if (result) {
        console.log(`\n[대상 ID 정보]`);
        console.log(`- ID: ${result.user_id}`);
        console.log(`- 이름: ${result.name}`);
        console.log(`- 나이: ${result.age}`);
        console.log(`- 도시: ${result.city}`);
    } else {
        console.log(`\n[대상 ID 정보를 찾을 수 없습니다: ${TARGET_ID}]`);
    }
}

function measureRepeatedSearch(rows, index) {
    let start;
    let end;

    start = process.hrtime.bigint();
    for (let i = 0; i < SEARCH_REPEAT; i++) {
        linearSearchFromFile(TARGET_ID);
    }
    end = process.hrtime.bigint();
    console.log(`파일 직접 선형 탐색 ${SEARCH_REPEAT}회: ${Number(end - start) / 1000000} ms`);

    start = process.hrtime.bigint();
    for (let i = 0; i < SEARCH_REPEAT; i++) {
        linearSearchFromRows(rows, TARGET_ID);
    }
    end = process.hrtime.bigint();
    console.log(`메모리 배열 선형 탐색 ${SEARCH_REPEAT}회: ${Number(end - start) / 1000000} ms`);

    start = process.hrtime.bigint();
    for (let i = 0; i < SEARCH_REPEAT; i++) {
        hashSearch(index, TARGET_ID);
    }
    end = process.hrtime.bigint();
    console.log(`해시 인덱스 탐색 ${SEARCH_REPEAT}회: ${Number(end - start) / 1000000} ms`);
}

function main() {
    const rows = parseCsvToArray();
    console.log(`총 레코드 수: ${rows.length}`);

    const index = measureIndexBuildCost(rows);
    measureSingleSearch(rows, index);
    measureRepeatedSearch(rows, index);
}

main();

Overwriting app.js


In [6]:
!node app.js

총 레코드 수: 300000
해시 인덱스 생성 시간: 311.414498 ms
파일 직접 선형 탐색 시간: 463.457771 ms
메모리 배열 선형 탐색 시간: 6.34002 ms
해시 인덱스 탐색 시간: 0.068499 ms

[대상 ID 정보]
- ID: user0300000
- 이름: 윤하은
- 나이: 20
- 도시: Seoul
파일 직접 선형 탐색 10회: 2782.69444 ms
메모리 배열 선형 탐색 10회: 59.062512 ms
해시 인덱스 탐색 10회: 0.006652 ms


# PART II

## 1. 실습명 : CSV 파일 기반 보안 로그 해시 인덱스 생성과 탐색 성능 비교

대용량 보안 로그 파일을 DB 없이 직접 처리해야 하는 침해 분석 및 이상 탐지 환경에서 사용된다. 파일 전체를 반복 탐색하는 구조는 비효율적이므로 메모리 기반 인덱스를 통해 성능을 개선한다. 해시 인덱스는 특정 사용자 또는 이벤트 ID 조회를 즉시 처리할 수 있어 반복 탐색 상황에서 효과적이다. 보안 로그 분석에서 빠른 추적을 위해 인덱스 구조를 코드 수준에서 이해하고 적용하기 위한 목적

## 2. 실습 목적

- Node.js에서 실제 보안 로그 CSV 파일을 읽는 과정을 통해 파일 기반 로그 처리 구조를 이해.
- 선형 탐색과 해시 인덱스 탐색을 각각 구현하여 보안 로그 검색 속도 차이를 확인.
- 인덱스 생성 비용과 탐지 속도 향상의 관계를 이해.
- 보안 로그 분석에서 빠른 이벤트 추적 구조를 학습.

## 3. 실습 개요

특정 사용자 ID 또는 이벤트 ID를 기반으로 로그를 찾는 기능을 두 가지 방식으로 구현하고 실행 시간을 비교

- 선형 탐색
- 해시 인덱스 탐색

In [3]:
%%writefile app.js

const fs = require('fs');
const path = require('path');

const FILE_PATH = path.join(__dirname, 'logs.csv');
const TARGET_ID = 'user099999';
const SEARCH_REPEAT = 10;

// CSV 파일 읽기
function readCsvLines() {
    // 파일 존재 여부 확인
    if (!fs.existsSync(FILE_PATH)) {
        throw new Error('logs.csv 파일이 없습니다.'); // 파일이 존재하지 않을 경우 즉시 오류 발생
    }
    // 파일이 존재하면 'utf8'형식으로 불러옴
    const content = fs.readFileSync(FILE_PATH, 'utf8');
    // 전체 내용을 문자열로 읽어 줄 단위 배열로 반환
    return content.trim().split('\n');
}

// CSV를 객체 배열로 반환
// readCsvLines()로 읽은 CSV 파일을 객체 배열로 반환하는 함수
function parseCsvToArray() {
    const lines = readCsvLines();
    const header = lines[0].split(','); //  헤더 -> 컬럼명 확인용

    const rows = [];

    // CSV 각 행을 자바스크립트 객체 배열로 변환
    for (let i = 1; i < lines.length; i++) {
        const cols = lines[i].split(',');
        rows.push({
            [header[0]]: cols[0],   // user_id
            [header[1]]: cols[1],   // event_type
            [header[2]]: cols[2],   // timestamp
            [header[3]]: cols[3]    // ip
        });
    }

    return rows;
}

// 해시 인덱스 생성
function buildHashIndex(rows) {
    const index = Object.create(null);

    for (let i = 0; i < rows.length; i++) {
        // user_id를 키로 사용하여 해시 인덱스 생성
        index[rows[i].user_id] = rows[i];
    }

    return index;
}

// 파일 직접 선형 탐색
function linearSearchFromFile(targetId) {
    // readCsvLines()로 CSV 파일 불러오기
    // 검색할 때마다 CSV 전체를 다시 읽고 분해하므로 가장 느린 탐색 구조
    const lines = readCsvLines();

    // lines 배열을 순회하며 모든 행을 처음부터 끝까지 한 줄씩 비교
    for (let i = 1; i < lines.length; i++) {
        const cols = lines[i].split(',');

        if (cols[0] === targetId) {
            return {
                user_id: cols[0],
                event_type: cols[1],
                timestamp: cols[2],
                ip: cols[3]
            };
        }
    }

    return null;
}

// 메모리 배열 선형 탐색
/* CSV 파일은 한 번만 읽고, 메모리에 적재된 배열을 앞에서부터 순차 탐색
   파일 직접 탐색보다 빠른 이유는 디스크 재접근이 제거되기 때문
   그러나 배열 전체를 처음부터 비교해야 하므로 데이터가 많아질수록 탐색 시간이 증가 */
function linearSearchFromRows(rows, targetId) {
    for (let i = 0; i < rows.length; i++) {
        if (rows[i].user_id === targetId) {
            return rows[i];
        }
    }

    return null;
}

// 해시 인덱스 탐색
/* 생성된 해시 인덱스에서 targetId를 키로 바로 접근
   반복문이 없어 가장 빠른 탐색 구조
   보안 로그에서 특정 사용자 ID를 즉시 조회할 때 사용하는 방식 */
function hashSearch(index, targetId) {
    return index[targetId] || null;
}

// 인덱스 생성 시간 측정
/* 해시 인덱스 생성하는 데 걸리는 시간을 측정
   인덱스는 검색 속도를 높이지만 미리 생성하는 비용이 필요함
   이러한 초기 비용을 감수하고도 해시 인덱스 탐색이 가장 빠른 탐색 방법일 경우 해당 방식 채택 */
function measureIndexBuildCost(rows) {
    const start = process.hrtime.bigint();
    const index = buildHashIndex(rows);
    const end = process.hrtime.bigint();

    console.log(`해시 인덱스 생성 시간: ${Number(end - start) / 1000000} ms`);
    return index;
}

// 단건 탐색 성능 비교
function measureSingleSearch(rows, index) {
    let start;
    let end;

    start = process.hrtime.bigint(); // process.hrtime.bigint()를 사용하여 짧은 시간 차이도 측정
    linearSearchFromFile(TARGET_ID);
    end = process.hrtime.bigint();
    console.log(`파일 직접 선형 탐색 시간: ${Number(end - start) / 1000000} ms`);

    start = process.hrtime.bigint();
    linearSearchFromRows(rows, TARGET_ID);
    end = process.hrtime.bigint();
    console.log(`메모리 배열 선형 탐색 시간: ${Number(end - start) / 1000000} ms`);

    start = process.hrtime.bigint();
    hashSearch(index, TARGET_ID);
    end = process.hrtime.bigint();
    console.log(`해시 인덱스 탐색 시간: ${Number(end - start) / 1000000} ms`);
}

// 반복 탐색 성능 비교
/* SEARCH_REPEAT 횟수 만큼 각 방법으로 반복 탐색
   반복 횟수 증가할수록 탐색 비용 증가 */
function measureRepeatedSearch(rows, index) {
    let start;
    let end;

    start = process.hrtime.bigint();
    for (let i = 0; i < SEARCH_REPEAT; i++) {
        linearSearchFromFile(TARGET_ID);
    }
    end = process.hrtime.bigint();
    console.log(`파일 직접 선형 탐색 ${SEARCH_REPEAT}회: ${Number(end - start) / 1000000} ms`);

    start = process.hrtime.bigint();
    for (let i = 0; i < SEARCH_REPEAT; i++) {
        linearSearchFromRows(rows, TARGET_ID);
    }
    end = process.hrtime.bigint();
    console.log(`메모리 배열 선형 탐색 ${SEARCH_REPEAT}회: ${Number(end - start) / 1000000} ms`);

    start = process.hrtime.bigint();
    for (let i = 0; i < SEARCH_REPEAT; i++) {
        hashSearch(index, TARGET_ID);
    }
    end = process.hrtime.bigint();
    console.log(`해시 인덱스 탐색 ${SEARCH_REPEAT}회: ${Number(end - start) / 1000000} ms`);
}

// 전체 실행 흐름
// 파일 읽기 -> 객체 배열 변환 -> 해시 인덱스 생성 -> 단건 비교 -> 반복 비교
function main() {
    const rows = parseCsvToArray();
    console.log(`총 로그 수: ${rows.length}`);

    const index = measureIndexBuildCost(rows);
    measureSingleSearch(rows, index);
    measureRepeatedSearch(rows, index);
}

main();

Overwriting app.js


In [4]:
!node app.js

총 로그 수: 100000
해시 인덱스 생성 시간: 165.0977 ms
파일 직접 선형 탐색 시간: 237.842438 ms
메모리 배열 선형 탐색 시간: 7.350112 ms
해시 인덱스 탐색 시간: 0.083102 ms
파일 직접 선형 탐색 10회: 1062.737269 ms
메모리 배열 선형 탐색 10회: 16.335912 ms
해시 인덱스 탐색 10회: 0.0085 ms


# PART III

## 1. 실습명 : CSV 파일 기반 디지털 서명 해시 검증과 위변조 탐지 성능 비교

대용량 파일 데이터의 무결성을 DB 없이 직접 검증해야 하는 보안 및 파일 검증 환경에서 사용된다. 파일 내용을 해시로 변환하여 저장하고 비교하는 방식은 데이터 위변조를 빠르게 탐지할 수 있다. 해시 기반 검증은 원본 데이터를 직접 비교하지 않고도 무결성을 확인할 수 있어 효율적이다. 디지털 서명의 핵심 원리인 해시 기반 무결성 검증 구조를 코드 수준에서 이해하고 적용하기 위한 목적

## 2. 실습 목적

- Node.js에서 파일 데이터를 읽고 해시를 생성하는 구조를 이해.
- 원본 데이터 비교 방식과 해시 비교 방식의 차이를 확인.
- 해시 기반 무결성 검증 구조를 이해.
- 디지털 서명에서 해시가 사용되는 이유를 학습.

## 3. 실습 개요

파일 또는 데이터의 위변조 여부를 두 가지 방식으로 검증하고 실행 시간을 비교

- 원본 데이터 직접 비교
- 해시 기반 비교

In [1]:
%%writefile app.js

// 모듈 불러오기
const fs = require('fs'); // CSV 파일을 읽는 데 사용
const path = require('path'); // 파일 경로를 안전하게 조합하는 데 사용
const crypto = require('crypto'); // SHA-256 해시를 생성하는 핵심 모듈

// 파일 경로와 실험 조건 설정
const FILE_PATH = path.join(__dirname, 'files.csv');
const TARGET_ID = 'file000012';
const SEARCH_REPEAT = 10;

// CSV 파일 읽기
function readCsvLines() {
    if (!fs.existsSync(FILE_PATH)) {
        throw new Error('files.csv 파일이 없습니다.');
    }

    const content = fs.readFileSync(FILE_PATH, 'utf8');
    return content.trim().split('\n');
}

// CSV를 객체 배열로 변환
function parseCsvToArray() {
    const lines = readCsvLines();
    const header = lines[0].split(',');

    const rows = [];

    for (let i = 1; i < lines.length; i++) {
        const cols = lines[i].split(',');
        rows.push({
            file_id: cols[0],
            content: cols[1],
            hash: cols[2]
        });
    }

    return rows;
}

// 해시 생성
function createHash(data) {
    return crypto.createHash('sha256').update(data).digest('hex');
}

// 해시 인덱스 생성
function buildHashIndex(rows) {
    const index = Object.create(null);

    for (let i = 0; i < rows.length; i++) {
        index[rows[i].file_id] = rows[i];
    }

    return index;
}

// 원본 데이터 직접 비교
function verifyByFullScan(targetId, targetContent) {
    const lines = readCsvLines();

    for (let i = 1; i < lines.length; i++) {
        const cols = lines[i].split(',');

        if (cols[0] === targetId) {
            return cols[1] === targetContent;
        }
    }

    return false;
}

// 해시 비교 방식
function verifyByHash(rows, targetId, targetContent) {
    for (let i = 0; i < rows.length; i++) {
        if (rows[i].file_id === targetId) {
            const newHash = createHash(targetContent);
            return rows[i].hash === newHash;
        }
    }

    return false;
}

// 해시 인덱스 기반 비교
function verifyByHashIndex(index, targetId, targetContent) {
    const row = index[targetId];
    if (!row) return false;

    const newHash = createHash(targetContent);
    return row.hash === newHash;
}

// 성능 측정
function measurePerformance(rows, index) {
    let start;
    let end;

    const targetContent = "sample_data_123";

    start = process.hrtime.bigint();
    verifyByFullScan(TARGET_ID, targetContent);
    end = process.hrtime.bigint();
    console.log(`원본 비교 방식: ${Number(end - start) / 1000000} ms`);

    start = process.hrtime.bigint();
    verifyByHash(rows, TARGET_ID, targetContent);
    end = process.hrtime.bigint();
    console.log(`해시 비교 방식: ${Number(end - start) / 1000000} ms`);

    start = process.hrtime.bigint();
    verifyByHashIndex(index, TARGET_ID, targetContent);
    end = process.hrtime.bigint();
    console.log(`해시 인덱스 방식: ${Number(end - start) / 1000000} ms`);
}

// 전체 실행 흐름
// CSV 읽기 -> 객체 배열 변환 -> 해시 인덱스 생성 -> 원본 비교 -> 해시 비교 -> 해시 인덱스 비교
function main() {
    const rows = parseCsvToArray();
    console.log(`총 파일 수: ${rows.length}`);

    const index = buildHashIndex(rows);
    measurePerformance(rows, index);
}

main();

Overwriting app.js


In [2]:
!node app.js

총 파일 수: 50000
원본 비교 방식: 68.477498 ms
해시 비교 방식: 1.005964 ms
해시 인덱스 방식: 0.091521 ms


# PART IV

## 1. 실습명 : 영상 파일 해시 탐색과 서비스 메타데이터 DB 연동 출력

대용량 영상 파일을 직접 저장소에서 관리하고, 서비스 정보는 별도 데이터베이스로 분리하여 운영하는 환경에서 사용된다. 영상 파일은 해시 기반 인덱스로 빠르게 탐색하고, 서비스 정보는 DB에서 조회하여 결합 출력하는 구조이다. 파일 탐색과 서비스 데이터 조회를 분리함으로써 성능과 확장성을 동시에 확보할 수 있다. 대규모 콘텐츠 플랫폼에서 사용하는 검색 구조를 코드 수준에서 이해하기 위한 목적

## 2. 실습 목적

- Node.js에서 영상 파일 목록 CSV를 읽고 해시 인덱스를 생성하는 구조를 이해.
- 해시 탐색으로 특정 영상 파일을 빠르게 찾는 방법을 학습.
- 영상 ID를 기준으로 데이터베이스에서 서비스 메타데이터를 조회하는 구조를 이해.
- 파일 저장소와 서비스 DB가 분리된 구조를 학습.

## 3. 실습 개요

영상 ID를 기준으로 파일 저장소에서 빠르게 탐색하고, 동일 ID로 DB에서 서비스 정보를 조회하여 결합 출력

- 해시 인덱스 기반 영상 파일 탐색
- 데이터베이스 기반 서비스 정보 조회

In [2]:
!npm install mysql2

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴
added 13 packages in 2s
⠴
⠴3 packages are looking for funding
⠴  run `npm fund` for details
⠴

In [3]:
%%writefile app.js

const fs = require('fs');
const path = require('path');
const mysql = require('mysql2/promise');

const FILE_PATH = path.join(__dirname, 'videos.csv');
const TARGET_ID = 'video000123';

const pool = mysql.createPool({
    host: 'localhost',
    user: 'testuser',
    password: '1234',
    database: 'testdb'
});

function readCsvLines() {
    if (!fs.existsSync(FILE_PATH)) {
        throw new Error('videos.csv 파일이 없습니다.');
    }

    const content = fs.readFileSync(FILE_PATH, 'utf8');
    return content.trim().split('\n');
}

function parseCsvToArray() {
    const lines = readCsvLines();
    const rows = [];

    for (let i = 1; i < lines.length; i++) {
        const cols = lines[i].split(',');
        rows.push({
            video_id: cols[0],
            file_path: cols[1],
            file_hash: cols[2]
        });
    }

    return rows;
}

function buildHashIndex(rows) {
    const index = Object.create(null);

    for (let i = 0; i < rows.length; i++) {
        index[rows[i].video_id] = rows[i];
    }

    return index;
}

function hashSearch(index, targetId) {
    return index[targetId] || null;
}

async function getVideoMetaFromDB(videoId) {
    const sql = `
        SELECT video_id, title, channel_name, views, created_at, status
        FROM video_metadata
        WHERE video_id = ?
    `;

    const [rows] = await pool.execute(sql, [videoId]);
    return rows.length > 0 ? rows[0] : null;
}

async function main() {
    const csvRows = parseCsvToArray();
    const index = buildHashIndex(csvRows);

    const videoFile = hashSearch(index, TARGET_ID);

    if (!videoFile) {
        console.log('영상 파일을 찾지 못했습니다.');
        return;
    }

    console.log('영상 파일 정보');
    console.log(videoFile);

    const videoMeta = await getVideoMetaFromDB(videoFile.video_id);

    if (!videoMeta) {
        console.log('영상 메타데이터를 찾지 못했습니다.');
        return;
    }

    console.log('영상 서비스 정보');
    console.log(videoMeta);

    const mergedResult = {
        video_id: videoFile.video_id,
        file_path: videoFile.file_path,
        file_hash: videoFile.file_hash,
        title: videoMeta.title,
        channel_name: videoMeta.channel_name,
        views: videoMeta.views,
        created_at: videoMeta.created_at,
        status: videoMeta.status
    };

    console.log('결합 출력 결과');
    console.log(mergedResult);

    await pool.end();
}

main().catch(function (err) {
    console.error(err);
});

Overwriting app.js


In [5]:
!node app.js

영상 파일 정보
{
  video_id: 'video000123',
  file_path: '/data/videos/video000123.mp4',
  file_hash: '437373784e8c720d923a568124fa757cb2bf10d7db86964874f059ae38c99453'
}
영상 서비스 정보
{
  video_id: 'video000123',
  title: 'Node.js 비동기 입문',
  channel_name: '코딩연구소',
  views: 12540,
  created_at: 2026-03-20T09:00:00.000Z,
  status: 'PUBLIC'
}
결합 출력 결과
{
  video_id: 'video000123',
  file_path: '/data/videos/video000123.mp4',
  file_hash: '437373784e8c720d923a568124fa757cb2bf10d7db86964874f059ae38c99453',
  title: 'Node.js 비동기 입문',
  channel_name: '코딩연구소',
  views: 12540,
  created_at: 2026-03-20T09:00:00.000Z,
  status: 'PUBLIC'
}


# **정리**

## 1. 파일 직접 선형 탐색 (Linear Search in File)

데이터베이스나 별도의 인덱스 없이, 디스크(SSD/HDD)에 저장된 파일의 첫 부분부터 끝까지 순차적으로 읽어나가며 원하는 데이터를 찾는 방식입니다.

**특징:**

* **저장 매체:** 보조 기억 장치 (Disk).
* **속도:** 가장 느림. 디스크 I/O(Input/Output)는 메모리 접근보다 수만 배 이상 느리기 때문입니다.
* **장점:** 별도의 준비 과정(정렬, 인덱스 생성)이 필요 없으며, 구현이 가장 단순합니다.
* **단점:** 데이터 양이 많아질수록 탐색 시간이 정비례하여 증가하며($O(n)$), 디스크 부하가 큽니다.

## 2. 메모리 배열 선형 탐색 (Linear Search in Memory Array)

데이터를 메모리(RAM) 상의 배열에 올려두고, 인덱스 0번부터 순차적으로 비교하며 찾는 방식입니다.

**특징:**

* **저장 매체:** 주 기억 장치 (RAM).
* **속도:** 파일 탐색보다는 훨씬 빠르지만, 알고리즘 복잡도는 여전히 $O(n)$입니다.
* **장점:** 데이터가 이미 메모리에 로드되어 있다면 디스크 접근 없이 빠르게 처리할 수 있습니다.
* **단점:** 데이터가 너무 많아 메모리 용량을 초과하면 사용할 수 없습니다. 데이터가 늘어날수록 성능이 선형적으로 저하됩니다.

## 3. 해시 인덱스 탐색 (Hash Index Search)

해시 함수를 사용하여 키(Key)를 특정 주소 값으로 변환하고, 해당 주소에 직접 접근하여 데이터를 찾는 방식입니다.

**특징:**
* **구조:**  키-값(Key-Value) 쌍으로 저장되는 해시 테이블 구조를 사용합니다.
* **속도:** 가장 빠름. 평균적인 시간 복잡도가 $O(1)$로, 데이터의 양과 상관없이 거의 즉시 데이터를 찾을 수 있습니다.
* **장점:** 대규모 데이터에서도 검색 성능이 비약적으로 우수합니다.
* **단점:** 해시 충돌(Collision) 관리가 필요하며, 범위를 검색(예: 10~20 사이 값 찾기)하거나 정렬된 순서로 읽는 데는 부적합합니다. 또한 인덱스를 저장하기 위한 추가 메모리 공간이 필요합니다.

## 탐색 방법별 특징 비교 요약

| 비교 항목 | 파일 직접 선형 탐색 | 메모리 배열 선형 탐색 | 해시 인덱스 탐색 |
| :--- | :--- | :--- | :--- |
| **저장 매체** | 보조기억장치 (HDD/SSD) | 주기억장치 (RAM) | RAM 또는 Disk (인덱스) |
| **시간 복잡도** | $O(n)$ | $O(n)$ | $O(1)$ (평균) |
| **탐색 속도** | **매우 느림** (디스크 I/O 발생) | **보통** (메모리 내 순차 접근) | **매우 빠름** (직접 주소 접근) |
| **장점** | 별도 준비 과정 없음, 구현 단순 | 구현이 쉽고 데이터 접근이 빠름 | 데이터 양과 무관하게 즉시 탐색 |
| **단점** | 데이터 증가 시 급격한 성능 저하 | 메모리 용량 제한, $O(n)$의 한계 | 추가 공간 필요, 범위 검색 불리 |
| **주요 용도** | 단순 로그 확인, 소량의 설정값 | 알고리즘 문제 풀이, 소규모 리스트 | 데이터베이스 인덱스, 캐시 시스템 |

## 해시(Hash)의 4가지 핵심 특징

해시 함수는 임의의 길이를 가진 데이터를 고정된 길이의 고유한 값(해시 값)으로 변환합니다.

1. **결정론적 (Deterministic):** 똑같은 입력값을 넣으면 항상 똑같은 결과가 나옵니다. (어제 넣든 내일 넣든 결과는 같습니다.)

2. **효율성 (Efficiency):** 아무리 큰 파일이라도 해시 값을 계산하는 속도가 매우 빠릅니다. 검색 속도는 평균적으로 $O(1)$에 수렴합니다.

3. **눈사태 효과 (Avalanche Effect):** 입력값이 아주 조금(점 하나만 찍어도)만 바뀌어도 결과값은 완전히 달라집니다. 데이터를 조작했는지 바로 알 수 있는 이유입니다.

4. **불가역성 (Irreversibility):** 해시 값만 보고 원래의 데이터가 무엇이었는지 역으로 계산해서 찾아내는 것이 거의 불가능합니다. (암호학적 해시의 특징)